<a href="https://colab.research.google.com/github/roughhawkbit/digi-inno-road-prod/blob/main/analysis/DataCharacterisationForPaper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
import math
import matplotlib
import numpy
import os
import pandas
import scipy
import sys

In [ ]:
if IN_COLAB:
  dirpath = '/content/digi-inno-road-prod'
  if not os.path.isdir(dirpath):
    # TODO git pull
    !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git
  sys.path.insert(0,dirpath)
else:
  module_path = os.path.abspath(os.path.join('..'))
  if not module_path in sys.path:
      sys.path.insert(0, module_path)

from innoprod.sheet_tools import get_sheet_dfs
from innoprod.text_analysis.chunking_tools import chunk_text_sentencewise
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps, wrangle_grants
from innoprod.wrangling.wrangling_tools import is_non_empty

In [ ]:
data = get_sheet_dfs()
roadmaps_df = wrangle_roadmaps(data['Roadmaps'])
grants_df = wrangle_grants(data['Grants'])

# Digital Readiness Score

In [ ]:
drs_col_name = 'Current Digital Readiness Score (refer to PAS:1040)'

DRS values were specified for this many firms:

In [ ]:
int(roadmaps_df[drs_col_name].notna().sum())

The mean DRS score:

In [ ]:
float(roadmaps_df[roadmaps_df[drs_col_name].notna()][drs_col_name].mean())

And the standard deviation:

In [ ]:
float(roadmaps_df[roadmaps_df[drs_col_name].notna()][drs_col_name].std())

Grouping these by DRS score:

In [ ]:
roadmaps_df[drs_col_name].value_counts().sort_index()

In [ ]:
ax = roadmaps_df[drs_col_name].value_counts().sort_index().plot.bar()
ax.set_ylabel('Number of firms')
ax.set_xlabel('Digital Readiness Score')
ax

## Attempting to fit the data to a binomial distrbution

In [ ]:
res = scipy.stats.fit(
    scipy.stats.binom,
    roadmaps_df[roadmaps_df[drs_col_name].notna()][drs_col_name],
    bounds={"n": [1,9]}
)
res.params

In [ ]:
# res.plot()

In [ ]:
nllf = res.nllf()
math.exp(-nllf)

## Attempting to fit to a normal distribution

In [ ]:
mu, sigma = scipy.stats.norm.fit(roadmaps_df[roadmaps_df[drs_col_name].notna()][drs_col_name])
print(f'mu={mu}, sigma={sigma}')

In [ ]:
res = scipy.stats.norm.fit(
    roadmaps_df[roadmaps_df[drs_col_name].notna()][drs_col_name]
)
res

In [ ]:
gof = scipy.stats.goodness_of_fit(
    scipy.stats.norm,
    roadmaps_df[roadmaps_df[drs_col_name].notna()][drs_col_name],
)
gof.pvalue

In [ ]:
scipy.stats.skewtest(roadmaps_df[roadmaps_df[drs_col_name].notna()][drs_col_name])

In [ ]:
scipy.stats.kurtosistest(roadmaps_df[roadmaps_df[drs_col_name].notna()][drs_col_name])

# Text

In [ ]:
qual_cols = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Details of any existing Digital Strategy', # Check: Rob added in on 21st May
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis'
]

short_labels = [
    "1. Edge Digital",
    "2. Investment barriers",
    "3. Existing strategy",
    "4. Strategic skills",
    "5. Technical skills",
    "6. Already investing?",
    "7. Identified problems"
]

In [ ]:
box_data = []
for i, col in enumerate(qual_cols):
  box_data.append(roadmaps_df[col].apply(lambda x: len(str(x).split())).to_numpy())

fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))
ax.boxplot(box_data)
ax.set_xticklabels(short_labels, rotation=30, ha="right")
ax.set_ylabel('Word count')

In [ ]:
roadmaps_df['Summary of the identified problems, including Gap Analysis'].apply(lambda x: len(str(x).split())).max()

In [ ]:
roadmaps_df['Summary of the identified problems, including Gap Analysis'].apply(lambda x: len(str(x).split())>500).value_counts()

In [ ]:
qual_df = roadmaps_df[['Client ID', 'Current Digital Readiness Score (refer to PAS:1040)'] + qual_cols].copy()
# qual_df = qual_df[roadmaps_df[drs_col_name].notna()]
qual_df['Context'] = roadmaps_df.apply(lambda row: ' '.join([row[c] for c in qual_cols]), axis=1)
qual_df.drop(qual_cols, axis=1, inplace=True)
qual_df["Word Count"] = qual_df["Context"].apply(lambda x: len(str(x).split()))
# qual_df

In [ ]:
step = 128
n_bins = 8
bins = []
weight_counts = {
    "With DRS": numpy.zeros(n_bins),
    "Without DRS": numpy.zeros(n_bins)
}

for i in range(0, n_bins*step, step):
  bins.append(f"{i}-{i+step-1}")
  weight_counts["With DRS"][int(i/step)] = qual_df[
      qual_df[drs_col_name].notna() &
      (qual_df['Word Count'] >= i) &
      (qual_df['Word Count'] < i+step)
    ]['Word Count'].count()
  weight_counts["Without DRS"][int(i/step)] = qual_df[
      qual_df[drs_col_name].isna() &
      (qual_df['Word Count'] >= i) &
      (qual_df['Word Count'] < i+step)
    ]['Word Count'].count()

bottom = numpy.zeros(len(bins))
width = 0.5
fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))

for drs_label, weight_count in weight_counts.items():
    p = ax.bar(bins, weight_count, width, label=drs_label, bottom=bottom)
    bottom += weight_count

ax.set_ylabel('Number of firms')
ax.set_xlabel('Word count')
ax.legend()
ax.set_xticklabels(bins, rotation=30)


Chunk the text that is longer than 510 words sentencewise, and see if this can split the longest text blocks into just two.

In [ ]:
qual_df['Chunked Text'] = qual_df['Context'].apply(lambda x: chunk_text_sentencewise(x, 510))
qual_df['N chunks'] = qual_df['Chunked Text'].apply(len)
qual_df['N chunks'].value_counts().sort_index()

In [ ]:
qual_df.plot.scatter(drs_col_name, 'Word count')